# Reflecting Brownian Motion

In [ ]:
# ================================================================
# Reflected Brownian Motion (RBM) simulation + Bayesian estimation
# ================================================================
# Author: ChatGPT
# Dependencies: jax, jaxlib, numpyro, matplotlib
# Install: pip install jax jaxlib numpyro matplotlib optax
# ================================================================

import jax
import jax.numpy as jnp
from jax import random, jit, vmap
import numpy as np
import matplotlib.pyplot as plt

import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, init_to_median


# ================================================================
# Simulation of reflected Brownian motion
# ================================================================
@jit
def simulate_rbm_single(key, sigma, x0, dt, nsteps):
    """Simulate one trajectory of reflected Brownian motion (RBM)."""
    sqrt_dt = jnp.sqrt(dt)

    def body(x, k):
        dW = random.normal(k) * sigma * sqrt_dt
        x_new = jnp.abs(x + dW)  # reflection at 0
        return x_new, x_new

    keys = random.split(key, nsteps)
    _, traj = jax.lax.scan(body, x0, keys)
    traj = jnp.concatenate([jnp.array([x0]), traj])
    return traj


@jit
def simulate_rbm_batch(key, sigma, x0, dt, nsteps, npaths):
    """Simulate many RBM trajectories in parallel, return their time-averages A_T."""
    keys = random.split(key, npaths)

    def simulate_one(k):
        traj = simulate_rbm_single(k, sigma, x0, dt, nsteps)
        t = jnp.linspace(0.0, nsteps * dt, nsteps + 1)
        A_T = jnp.trapz(traj, t) / (nsteps * dt)
        return A_T

    A_T = vmap(simulate_one)(keys)
    return A_T


@jit
def simulate_summaries(key, sigma, x0, dt, nsteps, npaths):
    """Compute mean and variance of A_T (time-averages)."""
    A_T = simulate_rbm_batch(key, sigma, x0, dt, nsteps, npaths)
    return jnp.mean(A_T), jnp.var(A_T), A_T


# ================================================================
# NumPyro model: synthetic likelihood for σ
# ================================================================
def synthetic_likelihood_model(obs_mean, obs_var, n_obs, *, x0, dt, nsteps, n_sim):
    """NumPyro model for Bayesian inference on σ (diffusion coefficient)."""
    sigma = numpyro.sample("sigma", dist.HalfNormal(2.0))  # prior for σ
    key = numpyro.prng_key()

    sim_mean, sim_var, _ = simulate_summaries(key, sigma, x0, dt, nsteps, n_sim)

    # Synthetic likelihood: assume obs_mean ~ N(sim_mean, sd_mean), obs_var ~ N(sim_var, sd_var)
    sd_mean = jnp.sqrt(jnp.maximum(sim_var / n_sim, 1e-6)) + 1e-3
    sd_var = jnp.maximum(0.5 * sim_var / jnp.sqrt(n_sim), 1e-3) + 1e-3

    numpyro.sample("obs_mean", dist.Normal(sim_mean, sd_mean), obs=obs_mean)
    numpyro.sample("obs_var", dist.Normal(sim_var, sd_var), obs=obs_var)


# ================================================================
# MCMC inference
# ================================================================
def run_inference(obs_A_samples, *, x0=0.0, dt=1e-3, T=5.0, n_sim=200, num_warmup=300, num_samples=600, rng_seed=0):
    """Run MCMC to infer σ from observed A_T samples."""
    obs_A = jnp.array(obs_A_samples)
    obs_mean = jnp.mean(obs_A)
    obs_var = jnp.var(obs_A)
    n_obs = obs_A.shape[0]
    nsteps = int(T / dt)

    model = lambda: synthetic_likelihood_model(
        obs_mean, obs_var, n_obs, x0=x0, dt=dt, nsteps=nsteps, n_sim=n_sim
    )

    kernel = NUTS(model, init_strategy=init_to_median())
    mcmc = MCMC(kernel, num_warmup=num_warmup, num_samples=num_samples, progress_bar=True)
    mcmc.run(jax.random.PRNGKey(rng_seed))
    mcmc.print_summary()
    return mcmc


# ================================================================
# Example run
# ================================================================
def example_run_small():
    key = random.PRNGKey(0)
    sigma_true = 1.0
    dt = 1e-3
    T = 5.0
    nsteps = int(T / dt)
    npaths_obs = 300  # number of observed trajectories
    key, sub = random.split(key)
    A_obs = simulate_rbm_batch(sub, sigma_true, x0=0.0, dt=dt, nsteps=nsteps, npaths=npaths_obs)

    print("Observed A_T mean, var:", float(jnp.mean(A_obs)), float(jnp.var(A_obs)))

    # Run MCMC inference for σ
    mcmc = run_inference(
        np.array(A_obs), x0=0.0, dt=dt, T=T, n_sim=200,
        num_warmup=200, num_samples=400, rng_seed=1
    )

    samples = mcmc.get_samples()
    sigma_samples = samples["sigma"]

    print("Posterior mean ± std:", float(jnp.mean(sigma_samples)), "±", float(jnp.std(sigma_samples)))

    plt.figure(figsize=(6, 3))
    plt.hist(sigma_samples, bins=40, density=True)
    plt.title("Posterior for σ (demo)")
    plt.xlabel("σ")
    plt.show()


# ================================================================
# Main
# ================================================================
if __name__ == "__main__":
    example_run_small()


# Multidimensional RBM

In [ ]:
import jax
import jax.numpy as jnp
from jax import random, jit

@jit
def simulate_rbm_multidim(key, mu, Sigma, R, x0, dt, nsteps):
    """
    Simulate a d-dimensional reflected Brownian motion (RBM).

    Args:
        key: jax.random.PRNGKey
        mu: drift vector (d,)
        Sigma: covariance matrix (d, d)
        R: reflection matrix (d, d)
        x0: initial condition (d,)
        dt: time step
        nsteps: number of steps
    Returns:
        traj: array of shape (nsteps+1, d)
    """
    d = x0.shape[0]
    chol = jnp.linalg.cholesky(Sigma)
    sqrt_dt = jnp.sqrt(dt)

    def body(x, k):
        z = random.normal(k, shape=(d,))
        dW = chol @ z * sqrt_dt
        y = x + mu * dt + dW
        # Compute boundary corrections
        delta_L = jnp.maximum(0.0, -y)
        x_new = y + R @ delta_L
        return x_new, x_new

    keys = random.split(key, nsteps)
    _, traj = jax.lax.scan(body, x0, keys)
    traj = jnp.vstack([x0, traj])
    return traj

# Example usage
if __name__ == "__main__":
    key = random.PRNGKey(0)
    mu = jnp.array([-0.1, 0.05])
    Sigma = jnp.array([[0.1, 0.05],
                       [0.05, 0.1]])
    R = jnp.array([[1.0, 0.0],
                   [0.2, 1.0]])  # lower-triangular reflection matrix
    x0 = jnp.array([0.5, 0.2])
    dt = 0.01
    nsteps = 10000

    traj = simulate_rbm_multidim(key, mu, Sigma, R, x0, dt, nsteps)
    print(traj[-10:])
